# 04 — Agentic Design Patterns (Code Examples)

Implementing all 4 patterns: Reflection, Tool Use, Planning, Multi-Agent.

**Prerequisites:** `pip install openai python-dotenv rich`

In [ ]:
import os, json, time
from dotenv import load_dotenv
from openai import OpenAI
from rich import print as rprint
from rich.panel import Panel

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
print('Setup complete')

## Pattern 1: Reflection — Generate → Critique → Improve

In [ ]:
def generate(task: str, context: str = "") -> str:
    """The Actor: generates initial output."""
    return client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Complete the task well." + (f" Previous feedback: {context}" if context else "")},
            {"role": "user", "content": task}
        ]
    ).choices[0].message.content

def critique(task: str, output: str) -> tuple[float, str]:
    """The Critic: evaluates and gives specific feedback."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content":
            f"Task: {task}\nOutput: {output}\n\n"
            "Score 0.0-1.0 and give ONE specific improvement suggestion.\n"
            "Format:\nSCORE: 0.X\nFEEDBACK: [one sentence]"
        }]
    ).choices[0].message.content
    
    score, feedback = 0.5, response
    for line in response.split('\n'):
        if line.startswith('SCORE:'):
            try: score = float(line.split(':')[1].strip())
            except: pass
        if line.startswith('FEEDBACK:'):
            feedback = line.replace('FEEDBACK:', '').strip()
    return score, feedback

def reflection_loop(task: str, threshold: float = 0.85, max_iters: int = 3) -> str:
    """Full reflection pattern: generate → critique → refine until quality threshold."""
    print(f"REFLECTION PATTERN")
    print(f"Task: {task}\n")
    
    feedback = ""
    for i in range(1, max_iters + 1):
        print(f"Iteration {i}:")
        output = generate(task, context=feedback)
        score, feedback = critique(task, output)
        print(f"  Output: {output[:100]}...")
        print(f"  Score: {score:.2f} | Feedback: {feedback[:80]}")
        
        if score >= threshold:
            print(f"  → Quality threshold reached!")
            return output
    
    print(f"  → Max iterations reached.")
    return output

result = reflection_loop(
    task="Explain gradient descent in 2 sentences. Must be clear enough for a 10-year-old.",
    threshold=0.85
)

## Pattern 2: Tool Use — Dynamic Tool Selection and Execution

In [ ]:
# Tool implementations
def web_search(query: str) -> str:
    mock = {
        "python": "Python 3.12 was released in October 2023 with significant performance improvements.",
        "langchain": "LangChain is the most popular framework for building LLM applications.",
        "default": f"Top results for '{query}': [AI frameworks, tutorials, and documentation]"
    }
    for k, v in mock.items():
        if k in query.lower():
            return v
    return mock["default"]

def calculator(expression: str) -> str:
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error: {e}"

def get_weather(city: str) -> str:
    return f"Weather in {city}: 22°C, partly cloudy. Wind: 15 km/h NW."

# Tool registry + OpenAI schemas
TOOL_FUNCTIONS = {"web_search": web_search, "calculator": calculator, "get_weather": get_weather}

TOOL_SCHEMAS = [
    {"type": "function", "function": {
        "name": "web_search",
        "description": "Search the web for current information about any topic.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "The search query"}
        }, "required": ["query"]}
    }},
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate a mathematical expression. Input must be a valid Python math expression.",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string", "description": "Math expression like '2 * 3 + 4'"}
        }, "required": ["expression"]}
    }},
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get current weather for a specific city.",
        "parameters": {"type": "object", "properties": {
            "city": {"type": "string", "description": "City name"}
        }, "required": ["city"]}
    }}
]

def tool_use_agent(user_query: str) -> str:
    """Agent that dynamically selects and uses the right tool."""
    print(f"TOOL USE PATTERN")
    print(f"Query: {user_query}")
    
    messages = [
        {"role": "system", "content": "You are an agent. Use tools to answer accurately."},
        {"role": "user",   "content": user_query}
    ]
    
    # Allow multiple tool calls in sequence
    for step in range(5):
        response = client.chat.completions.create(
            model="gpt-4o-mini", messages=messages, tools=TOOL_SCHEMAS, tool_choice="auto"
        )
        msg = response.choices[0].message
        
        if not msg.tool_calls:
            print(f"Answer: {msg.content}")
            return msg.content
        
        # Execute all tool calls in this step
        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            print(f"  Tool: {tc.function.name}({args})")
            result = TOOL_FUNCTIONS[tc.function.name](**args)
            print(f"  Result: {result}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

# Test with different queries — agent picks the right tool automatically
print("=" * 55)
tool_use_agent("What is 15% of 847 plus 23 cubed?")
print()
tool_use_agent("What's the weather like in Tokyo?")
print()
tool_use_agent("What is LangChain used for?")

## Pattern 3: Planning — Plan Upfront, Execute, Re-plan on Failure

In [ ]:
def create_plan(goal: str, available_tools: list[str]) -> list[dict]:
    """Generate a structured plan as a list of steps."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": (
                f"Goal: {goal}\n"
                f"Available tools: {available_tools}\n\n"
                "Create a step-by-step plan as JSON. Format:\n"
                '[{"step": 1, "action": "tool_name", "description": "what to do", "expected_output": "what you expect"}]\n'
                "Max 4 steps. Return ONLY valid JSON."
            )
        }]
    ).choices[0].message.content
    
    try:
        # Extract JSON if wrapped in code blocks
        if '```' in response:
            response = response.split('```')[1].replace('json', '').strip()
        return json.loads(response)
    except:
        return [{"step": 1, "action": "web_search", "description": goal, "expected_output": "information"}]

def execute_step(step: dict) -> tuple[bool, str]:
    """Execute a plan step. Returns (success, result)."""
    action = step.get("action", "web_search")
    desc = step["description"]
    
    if action in TOOL_FUNCTIONS:
        try:
            # Determine args based on tool
            if action == "web_search":
                result = web_search(desc)
            elif action == "calculator":
                result = calculator(desc)
            elif action == "get_weather":
                result = get_weather(desc)
            else:
                result = f"Executed: {desc}"
            return True, result
        except Exception as e:
            return False, str(e)
    else:
        return True, f"Completed: {desc}"

def planning_agent(goal: str):
    """Full plan-and-execute agent with re-planning on failure."""
    print("PLANNING PATTERN")
    print(f"Goal: {goal}")
    
    tools = list(TOOL_FUNCTIONS.keys())
    
    plan = create_plan(goal, tools)
    print(f"\nPlan created ({len(plan)} steps):")
    for s in plan:
        print(f"  Step {s['step']}: [{s['action']}] {s['description'][:60]}")
    
    results = []
    print("\nExecuting plan:")
    for step in plan:
        print(f"  Executing step {step['step']}...")
        success, result = execute_step(step)
        
        if success:
            print(f"  ✅ {result[:80]}")
            results.append(result)
        else:
            print(f"  ❌ Step failed: {result}")
            print(f"  Re-planning from step {step['step']}...")
            # Re-planning: create new plan from current state
            remaining_goal = f"Starting from: {results}. Now: {goal}"
            plan = create_plan(remaining_goal, tools)
            print(f"  New plan has {len(plan)} steps")
    
    print(f"\nPlan complete. {len(results)} steps succeeded.")

planning_agent("Research the top 3 AI agent frameworks and summarize their differences")

## Pattern 4: Multi-Agent — Specialist Agents with a Coordinator

In [ ]:
def specialist_agent(role: str, task: str, tools: list | None = None) -> str:
    """A specialist agent with a specific role and optional tools."""
    system_prompts = {
        "researcher": "You are an expert researcher. Find accurate, detailed information. Be thorough.",
        "writer":     "You are an expert content writer. Write clear, engaging, well-structured content.",
        "critic":     "You are a critical reviewer. Identify gaps, errors, and areas for improvement. Be specific.",
        "summarizer": "You are an expert at summarizing. Create concise, accurate summaries preserving key points.",
    }
    
    messages = [
        {"role": "system", "content": system_prompts.get(role, "You are a helpful AI agent.")},
        {"role": "user",   "content": task}
    ]
    
    kwargs = {"model": "gpt-4o-mini", "messages": messages}
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"
    
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content

def coordinator(goal: str) -> dict:
    """Coordinator: decides which agents to use and what order."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content":
            f"Goal: {goal}\nAvailable agents: researcher, writer, critic, summarizer\n\n"
            "Which agents should work on this and in what order? Give each agent a specific sub-task.\n"
            'Return JSON: [{"agent": "name", "task": "specific task description"}]\n'
            "Max 3 agents. Return ONLY valid JSON."
        }]
    ).choices[0].message.content
    
    try:
        if '```' in response:
            response = response.split('```')[1].replace('json', '').strip()
        return json.loads(response)
    except:
        return [{"agent": "researcher", "task": goal}]

def multi_agent_system(goal: str):
    """Multi-agent pipeline: coordinator delegates to specialists."""
    print("MULTI-AGENT PATTERN")
    print(f"Goal: {goal}")
    
    # Step 1: Coordinator plans which agents do what
    plan = coordinator(goal)
    print(f"\nCoordinator assigned {len(plan)} agents:")
    for p in plan:
        print(f"  → {p['agent']:12}: {p['task'][:65]}")
    
    # Step 2: Execute agents sequentially, passing results forward
    accumulated_context = ""
    print("\nExecuting agents:")
    
    for assignment in plan:
        agent_role = assignment["agent"]
        task = assignment["task"]
        
        if accumulated_context:
            task = f"Previous work:\n{accumulated_context}\n\nYour task: {task}"
        
        print(f"\n  [{agent_role.upper()}] working...")
        result = specialist_agent(agent_role, task)
        print(f"  Output: {result[:120]}...")
        
        accumulated_context += f"\n\n[{agent_role}]: {result}"
    
    print("\nAll agents complete. Final output from last agent shown above.")

multi_agent_system(
    "Create a short explanation of what LangGraph is and why engineers should use it"
)

## Key Takeaways — All 4 Patterns

In [ ]:
from rich.table import Table

table = Table(title="4 Agentic Design Patterns")
table.add_column("Pattern",      style="bold cyan", width=15)
table.add_column("Core Loop",    width=30)
table.add_column("When to Use",  width=30)
table.add_column("Cost",         width=10)

for row in [
    ("Reflection",   "Generate → Critique → Refine", "Quality-critical outputs",          "2–5× calls"),
    ("Tool Use",     "Perceive → Select Tool → Act", "Any task needing external data",    "1–3× calls"),
    ("Planning",     "Plan → Execute → Re-plan",     "Complex multi-step goals",          "N+1 calls"),
    ("Multi-Agent",  "Coord → Delegate → Combine",   "Tasks needing specialization",      "N×M calls"),
]:
    table.add_row(*row)

rprint(table)
print("\nIn production: combine all 4 for maximum capability.")